<a href="https://colab.research.google.com/github/rah-ds/Cloud-Autoscaling-using-RL/blob/bmcgregor%2Fsimulator-refactor/Experiment_REINFORCE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

df_usage = pd.read_csv('/content/drive/MyDrive/df_usage.csv')
display(df_usage.head())

,time_window,avg_cpu,avg_mem,active_machines
0,1970-01-01 00:05:00+00:00,0.006623,0.004912,9525
1,1970-01-01 00:06:00+00:00,0.003254,0.002733,3805
2,1970-01-01 00:07:00+00:00,0.003070,0.002770,4167
3,1970-01-01 00:08:00+00:00,0.001950,0.001823,4338
4,1970-01-01 00:09:00+00:00,0.001689,0.001468,5545


In [4]:
import requests

raw_url = "https://raw.githubusercontent.com/rah-ds/Cloud-Autoscaling-using-RL/bmcgregor/simulator-refactor/scripts/autoscaling_env.py"
local_filename = "autoscaling_env.py"

r = requests.get(raw_url)
r.raise_for_status()  # ensures you get an error if download fails

with open(local_filename, "wb") as f:
    f.write(r.content)

print("Downloaded:", local_filename)


Downloaded: autoscaling_env.py


In [5]:
import importlib
import autoscaling_env # Ensure the module is loaded if not already

# Reload the module to pick up the changes
importlib.reload(autoscaling_env)

# Re-import the class and re-initialize the environment
from autoscaling_env import AutoScalingEnv
env = AutoScalingEnv(df_usage)

print('Resetting the environment...')
initial_state, initial_info = env.reset()
print(f'Initial State: {initial_state}')
print(f'Initial Info: {initial_info}')

import random
action = random.randint(0, env.action_space.n - 1)
print(f'Taking random action: {action}')

next_state, reward, terminated, truncated, info = env.step(action)

print(f'\nAfter one step:')
print(f'Next State: {next_state}')
print(f'Reward: {reward}')
print(f'Terminated: {terminated}')
print(f'Truncated: {truncated}')
print(f'Info: {info}')

Resetting the environment...
Initial State: [6.6225836e-03 4.9124165e-03 1.0000000e+01]
Initial Info: {'initial_capacity': 10}
Taking random action: 0

After one step:
Next State: [3.2542923e-03 2.7331815e-03 9.0000000e+00]
Reward: -68.51590861466157
Terminated: False
Truncated: False
Info: {'current_capacity': 9, 'utilization': np.float64(7.008900783151052), 'estimated_total_cpu_load': np.float64(63.080107048359466), 'reward_components': {'cost_penalty': -0.018000000000000002, 'sla_penalty': np.float64(-62.08900783151052), 'util_deviation_penalty': np.float64(-6.4089007831510525)}}


### 1. Define the Policy Network
We'll create a simple neural network using TensorFlow/Keras to serve as our policy. This network will map states to action probabilities.

In [6]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np

# Get observation and action space dimensions from the environment
observation_space_dim = env.observation_space.shape[0]
action_space_dim = env.action_space.n

# Define the Policy Network
def create_policy_model(obs_dim, action_dim):
    model = models.Sequential([
        layers.Input(shape=(obs_dim,)),
        layers.Dense(128, activation='relu'),
        layers.Dense(128, activation='relu'),
        layers.Dense(action_dim, activation='softmax') # Softmax for probabilities
    ])
    return model

policy_model = create_policy_model(observation_space_dim, action_space_dim)
policy_model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,411 (68.01 KB)

 Trainable params: 17,411 (68.01 KB)

 Non-trainable params: 0 (0.00 B)

### 2. Implement the REINFORCE Algorithm
Now, let's define the REINFORCE algorithm. This involves:
- Collecting trajectories (sequences of states, actions, and rewards) from the environment.
- Calculating the discounted return for each step in the trajectory.
- Updating the policy network using gradient ascent, aiming to increase the probability of actions that led to higher returns.

In [7]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

def discount_rewards(rewards, gamma=0.99):
    discounted_r = np.zeros_like(rewards, dtype=np.float32)
    running_add = 0
    for t in reversed(range(0, len(rewards))):
        running_add = running_add * gamma + rewards[t]
        discounted_r[t] = running_add
    return discounted_r

def train_step(states, actions, returns):
    with tf.GradientTape() as tape:
        # Get action probabilities from the policy model
        action_probabilities = policy_model(states)

        # Select probabilities corresponding to the taken actions
        indices = tf.stack([tf.range(len(actions)), tf.cast(actions, tf.int32)], axis=1)
        selected_action_probs = tf.gather_nd(action_probabilities, indices)

        # Compute the loss (negative log probability weighted by return)
        # Add a small epsilon to log to prevent log(0)
        loss = -tf.reduce_sum(tf.math.log(selected_action_probs + 1e-8) * returns)

    # Compute gradients and apply them
    gradients = tape.gradient(loss, policy_model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, policy_model.trainable_variables))
    return loss


### 3. Training Loop
We will now run a number of episodes to train the REINFORCE agent. We'll collect data for each episode and use it to update the policy network.

In [ ]:
num_episodes = 200
gamma = 0.99 # Discount factor

episode_rewards = []

for episode in range(num_episodes):
    states, actions, rewards = [], [], []
    state, info = env.reset()
    done = False
    episode_reward = 0

    while not done:
        # Reshape state to fit model input (batch size 1)
        state_input = tf.convert_to_tensor(state[np.newaxis, :], dtype=tf.float32)

        # Predict action probabilities
        action_probs = policy_model(state_input).numpy()[0]

        # Sample an action from the distribution
        action = np.random.choice(action_space_dim, p=action_probs)

        # Take action in the environment
        next_state, reward, terminated, truncated, info = env.step(action)

        done = terminated or truncated

        states.append(state)
        actions.append(action)
        rewards.append(reward)

        state = next_state
        episode_reward += reward

    # Convert lists to arrays
    states = np.array(states, dtype=np.float32)
    actions = np.array(actions, dtype=np.int32)
    rewards = np.array(rewards, dtype=np.float32)

    # Calculate discounted returns
    returns = discount_rewards(rewards, gamma)

    # Normalize returns (optional, but often helps training stability)
    returns = (returns - np.mean(returns)) / (np.std(returns) + 1e-8)

    # Perform a training step
    loss = train_step(states, actions, returns)

    episode_rewards.append(episode_reward)

    if episode % 10 == 0:
        print(f"Episode {episode}: Total Reward = {episode_reward:.2f}, Loss = {loss:.4f}, Avg Reward (last 10) = {np.mean(episode_rewards[-10:]):.2f}")

print("\nTraining finished!")

Episode 0: Total Reward = -859913.48, Loss = -143.5439, Avg Reward (last 10) = -859913.48
Episode 10: Total Reward = -14030.56, Loss = -91.3181, Avg Reward (last 10) = -14092.76


### 4. Evaluate the Trained Agent
Let's visualize the training progress and run a final evaluation episode to see how the agent performs.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.plot(episode_rewards)
plt.title('REINFORCE Training: Episode Rewards Over Time')
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.grid(True)
plt.show()

print("\n--- Evaluation of Trained Agent ---")
state, info = env.reset()
done = False
episode_reward = 0

while not done:
    state_input = tf.convert_to_tensor(state[np.newaxis, :], dtype=tf.float32)
    action_probs = policy_model(state_input).numpy()[0]
    action = np.argmax(action_probs) # Take the most probable action for evaluation

    next_state, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated

    state = next_state
    episode_reward += reward

print(f"Evaluation Episode: Total Reward = {episode_reward:.2f}")
